# Thermodynamics from the phonon spectrum

Compare nine quantities calculated from the unperturbed NCrystal graphite PDOS and three stochastic realizations. The reference and samples use the same TARA integrals, temperature, mass, and unit-area PDOS convention.

## 1. Locate TARA

The next cell finds TARA and imports the PDOS APIs.

In [ ]:
import sys
from pathlib import Path

import NCrystal as NC
import numpy as np

PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "core" / "config.py").is_file()
     and (candidate / "analyzer" / "physics" / "thermodynamics.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside the TARA repository.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from core.config import TaraConfig
from core.data.input_pdos import PhononSpectra
from core.sampler.concrete_samplers import TaraSampler
from input_output.ncrystal.core import NCrystalFile

print(f"TARA project: {PROJECT_ROOT}")

## 2. Select the required inputs

| Setting | Example value | What it controls |
| --- | --- | --- |
| `BASE_NCMAT_FILE` | `tests/fixtures/C_sg194_pyrolytic_graphite.ncmat` | NCrystal graphite template. |

In [ ]:
fixtures = PROJECT_ROOT / "tests" / "fixtures"
BASE_NCMAT_FILE = fixtures / "C_sg194_pyrolytic_graphite.ncmat"
if not BASE_NCMAT_FILE.is_file():
    raise FileNotFoundError(f"NCMAT template missing: {BASE_NCMAT_FILE}")

## 3. Load the graphite reference

**Fixed settings for this example**

| Setting | Example value | What it controls |
| --- | --- | --- |
| `TEMPERATURE_K` | `293.6` | Evaluation temperature in kelvin. |
| `ATOM_LABEL` | `"C"` | Species supplying the reference spectrum. |

In [ ]:
TEMPERATURE_K = 293.6
ATOM_LABEL = "C"
print("Engine: NCrystal")
print(f"temperature: {TEMPERATURE_K:g} K")

material = NCrystalFile.from_file(str(BASE_NCMAT_FILE))
raw_pdos = material.pdos_for_atom(ATOM_LABEL)
reference_spectrum = PhononSpectra(pdos=raw_pdos)
MASS_AMU = NC.atomDB(ATOM_LABEL).averageMassAMU()
print(f"NCMAT file: {BASE_NCMAT_FILE}")

## 4. Sample the graphite phonon spectrum

**Required:** The PDOS from Section 3 and the two perturbation bounds below. Select a positive sample count for the comparison.

| Setting | Example value | What it controls |
| --- | --- | --- |
| `MAX_ENERGY_SHIFT_MEV` | `1.0` | Maximum phonon-energy displacement in meV. |
| `MAX_AMPLITUDE_SHIFT_PCT` | `5.0` | Maximum relative PDOS amplitude change in percent. |
| `N_SAMPLES` | `3` | Number of stochastic samples. The reference is stored separately. |

In [ ]:
MAX_ENERGY_SHIFT_MEV = 1.0
MAX_AMPLITUDE_SHIFT_PCT = 5.0
N_SAMPLES = 3

**Optional sampling settings**

| Setting | Example value | What it controls |
| --- | --- | --- |
| `RANDOM_SEED` | `20261008` | Seed for reproducible sampling. |
| `BATCH_SIZE` | `512` | Internal sampling chunk size. |
| `AMPLITUDE_METHOD` | `"fast"` | Method used for the amplitude perturbations. |
| `SPATIAL_METHOD` | `"diffeomorphic"` | Method used to shift the spectrum in energy. |

In [ ]:
RANDOM_SEED = 20261008
BATCH_SIZE = 512
AMPLITUDE_METHOD = "fast"
SPATIAL_METHOD = "diffeomorphic"

In [ ]:
# Validate all perturbation bounds and sampling strategies using TaraConfig.
configuration = TaraConfig(
    max_energy_shift_mev=MAX_ENERGY_SHIFT_MEV,
    max_amplitude_shift_pct=MAX_AMPLITUDE_SHIFT_PCT,
    n_samples=N_SAMPLES,
    random_seed=RANDOM_SEED,
)

# Initialize TaraSampler with the selected amplitude and spatial methods.
sampler = TaraSampler(
    config=configuration,
    spectra=reference_spectrum,
    amplitude_method=AMPLITUDE_METHOD,
    spatial_method=SPATIAL_METHOD,
)

# Generate the stochastic ensemble with the configured sampling batch size.
ensemble = sampler.generate_ensemble(batch_size=BATCH_SIZE)
if ensemble.reference is None or len(ensemble) != N_SAMPLES:
    raise RuntimeError("Expected the reference and all requested samples.")
print(f"Prepared one reference and {len(ensemble)} stochastic spectra.")

## 5. Select the thermodynamic quantities

Use the same temperature and mass for the reference and samples.

**Optional setting**

| Setting | Example value | What it controls |
| --- | --- | --- |
| `PHONON_MOMENT_ORDER` | `2` | Order of the PDOS power moment. |

In [ ]:
PHONON_MOMENT_ORDER = 2

## 6. Evaluate the reference and samples

**Required:** Run the same calculations on each unit-area PDOS. The baseline values come from applying these integrals to the original NCrystal spectrum.

In [ ]:
from analyzer.physics.thermodynamics import (
    calc_cv_numba,
    calc_entropy_numba,
    calc_teff_numba,
    calc_msd_numba,
    calc_zero_point_energy_numba,
    calc_free_energy_numba,
    calc_internal_energy_numba,
    calc_log_moment_numba,
    calc_phonon_moment_numba,
)

def evaluate_thermodynamics(spectrum):
    """Evaluate the selected quantities for one unit-area PDOS."""
    energy = spectrum.energy_grid
    density = spectrum.density_values
    return np.array([
        calc_cv_numba(density, energy, TEMPERATURE_K),
        calc_entropy_numba(density, energy, TEMPERATURE_K),
        calc_teff_numba(density, energy, TEMPERATURE_K),
        calc_msd_numba(density, energy, TEMPERATURE_K, MASS_AMU),
        calc_zero_point_energy_numba(density, energy),
        calc_free_energy_numba(density, energy, TEMPERATURE_K),
        calc_internal_energy_numba(density, energy, TEMPERATURE_K),
        calc_log_moment_numba(density, energy),
        calc_phonon_moment_numba(density, energy, PHONON_MOMENT_ORDER),
    ])


reference_values = evaluate_thermodynamics(ensemble.reference)
sample_values = np.stack([evaluate_thermodynamics(sample) for sample in ensemble])
if not np.isfinite(reference_values).all() or not np.isfinite(sample_values).all():
    raise RuntimeError("Expected finite thermodynamic values for the reference and samples.")

## 7. Plot the effect of sampling

Each panel shows the three sample values as colored dots and the NCrystal PDOS baseline as a horizontal line. The vertical scales are set separately for each quantity so the sampling effects are visible. The heat capacity kernel reports the value in units of Boltzmann’s constant.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

titles = [
    "Heat capacity",
    "Entropy",
    "Effective temperature",
    "Mean-square displacement",
    "Zero-point energy",
    "Free energy",
    "Internal energy",
    "Logarithmic moment",
    f"PDOS moment (order {PHONON_MOMENT_ORDER})",
]

# Positive values indicate a smaller sample-to-reference ratio.
sample_to_reference = np.divide(
    sample_values,
    reference_values,
    out=np.full(sample_values.shape, np.nan),
    where=reference_values != 0.0,
)
difference_pct = 100.0 * (1.0 - sample_to_reference)

sample_ids = np.arange(1, len(sample_values) + 1)
colors = [f"C{i % 10}" for i in range(len(sample_ids))]

fig, axes = plt.subplots(
    3, 3, figsize=(11, 8.3), sharex=True, layout="constrained"
)

for index, (ax, title) in enumerate(zip(axes.flat, titles)):
    ax.set(title=title, ylabel="Relative difference (%)")
    ax.set_xticks(sample_ids)
    ax.set_xlim(0.6, len(sample_ids) + 0.4)
    ax.grid(axis="y", alpha=0.22)

    if reference_values[index] == 0.0:
        ax.text(
            0.5, 0.5, "Undefined for a zero reference",
            transform=ax.transAxes, ha="center", va="center",
        )
        continue

    ax.axhline(0, color="black", linewidth=1.3, label="NCrystal reference")
    ax.scatter(sample_ids, difference_pct[:, index], c=colors, s=45, zorder=3)

    if index == 0:
        ax.legend(loc="best", frameon=False, fontsize=9)

fig.supxlabel("Stochastic sample")
fig.suptitle(f"Effect of graphite PDOS sampling at {TEMPERATURE_K:g} K")
plt.show()